# 01. Baseline pipeline (skeleton)

Preliminary assessment of skin conditions from a photo. **Decision support, not a diagnosis.**
This notebook is a skeleton: it contains **no results**. Every `TODO` must be filled with real data and measured numbers.

Related: `docs/04-technology-comparison.md`, `docs/08-verification-evaluation-plan.md`, `ml/README.md`.

## 0. Config

In [ ]:
SEED = 42
DATA_DIR = "../data"          # TODO: place datasets here (ml/data is git-ignored)
SPLIT_DIR = "../splits"       # committed split files
IMG_SIZE = 224                # TODO: confirm for the chosen backbone
BATCH_SIZE = 32               # TODO
EPOCHS = 0                    # TODO: set when training for real
CLASSES = []                  # TODO: planned 5-10 classes, list pending supervisor's answer
DANGEROUS_CLASSES = []        # TODO: agree with supervisor or medical expert

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix, roc_auc_score

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Data

Datasets from `docs/02`: [3] HAM10000, [7] Fitzpatrick 17k, [9] DDI. Check terms of use first (see `ml/README.md`).
Expected metadata table columns: `image_path`, `label`, `patient_id` (or `lesion_id`), `source`, optional `skin_tone`.

`TODO:` write one loader per dataset that returns this table, then concatenate. Split first, merge after.

In [ ]:
def load_metadata() -> pd.DataFrame:
    """TODO: build the metadata table from the datasets in DATA_DIR."""
    raise NotImplementedError("TODO: implement dataset loading")

# df = load_metadata()
# df.head()

## 2. Patient-level split

All images of one patient (or lesion, if there is no patient ID) must be in the same subset. Save the split to `SPLIT_DIR` and commit it.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def patient_level_split(df, group_col="patient_id", test_size=0.15, val_size=0.15, seed=SEED):
    groups = df[group_col]
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trainval_idx, test_idx = next(gss.split(df, groups=groups))
    trainval = df.iloc[trainval_idx]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size / (1 - test_size), random_state=seed)
    train_idx, val_idx = next(gss2.split(trainval, groups=trainval[group_col]))
    train, val, test = trainval.iloc[train_idx], trainval.iloc[val_idx], df.iloc[test_idx]
    # sanity check: no patient appears in two subsets
    assert not (set(train[group_col]) & set(val[group_col]))
    assert not (set(train[group_col]) & set(test[group_col]))
    assert not (set(val[group_col]) & set(test[group_col]))
    return train, val, test

# train_df, val_df, test_df = patient_level_split(df)
# train_df.to_csv(f"{SPLIT_DIR}/train.csv", index=False); ...

## 3. Dataset and transforms

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),   # TODO: tune after inspecting data
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class SkinDataset(Dataset):
    def __init__(self, df, tf):
        self.df, self.tf = df.reset_index(drop=True), tf
        self.class_to_idx = {c: i for i, c in enumerate(CLASSES)}
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["image_path"]).convert("RGB")
        return self.tf(img), self.class_to_idx[row["label"]]

## 4. Model

Draft hypothesis (`docs/04`, Decision 1): compact pretrained CNN with transfer learning, candidate MobileNetV3 [4]. `TODO: confirm by measurement; compare with a simple baseline and with alternatives.`

In [ ]:
def build_model(num_classes: int) -> nn.Module:
    # TODO: backbone is a draft choice (docs/04), not locked
    model = models.mobilenet_v3_small(weights="DEFAULT")
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model

# model = build_model(len(CLASSES)).to(device)

## 5. Training

`TODO:` class-imbalance handling (weighted loss or sampling), early stopping on validation, logging of seed, split file and commit. Save checkpoints to `ml/checkpoints/` (git-ignored).

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

# TODO: training loop, validation, checkpointing (runs/ is not committed)

## 6. Evaluation

Primary: recall per class, dangerous classes first. Then specificity, precision, F1, AUROC with bootstrap CI over patients, confusion matrix, calibration. Stratify by image source and skin tone where labels exist.
The test set is used once per reported experiment.

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, ys = [], []
    for x, y in loader:
        probs.append(torch.softmax(model(x.to(device)), dim=1).cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(probs), np.concatenate(ys)

def per_class_report(y_true, probs, classes, dangerous):
    y_pred = probs.argmax(1)
    rows = []
    for i, c in enumerate(classes):
        rec = recall_score(y_true == i, y_pred == i, zero_division=0)
        pre = precision_score(y_true == i, y_pred == i, zero_division=0)
        f1 = f1_score(y_true == i, y_pred == i, zero_division=0)
        rows.append({"class": c, "recall": rec, "precision": pre, "f1": f1, "dangerous": c in dangerous})
    return pd.DataFrame(rows), confusion_matrix(y_true, y_pred)

# TODO: AUROC per class with bootstrap CI over patients; calibration (ECE)

## 7. Phone-photo check

Separate set, reported separately from the benchmark test set. `TODO:` source of photos and consent (see `docs/08`).

## 8. Export and on-device measurements

`TODO: measure` model size, latency on a mid-range phone, recall before and after compression.

## Results

_None yet. Add only numbers produced by a logged run (commit, seed, split file, date)._